In [1]:
import tensorflow as tf
import numpy as np
from pathlib import Path

# Project paths
MODEL_DIR = Path("../models")
PROCESSED_DIR = Path("../data/processed")

# Load the trained baseline LSTM
model = tf.keras.models.load_model(
    MODEL_DIR / "baseline_lstm.keras"
)

# Load test data
X_test_lstm = np.load(PROCESSED_DIR / "X_test_lstm.npy")
y_test_lstm = np.load(PROCESSED_DIR / "y_test_lstm.npy")

print("Baseline model and test data loaded successfully")
print("=" * 55)
print("Test data shape :", X_test_lstm.shape)
print("Test labels     :", y_test_lstm.shape)

Baseline model and test data loaded successfully
Test data shape : (45000, 78, 1)
Test labels     : (45000,)


In [2]:
def fgsm_attack(model, x, y, epsilon):
    """
    Generate an adversarial example using FGSM.

    epsilon controls the perturbation strength.
    """

    x_tensor = tf.convert_to_tensor(x, dtype=tf.float32)
    y_tensor = tf.convert_to_tensor(
        y.reshape(-1, 1),
        dtype=tf.float32
    )

    with tf.GradientTape() as tape:
        tape.watch(x_tensor)

        predictions = model(x_tensor, training=False)

        loss = tf.keras.losses.binary_crossentropy(
            y_tensor,
            predictions
        )

        loss = tf.reduce_mean(loss)

    gradient = tape.gradient(loss, x_tensor)

    # FGSM perturbation
    perturbation = epsilon * tf.sign(gradient)

    x_adversarial = x_tensor + perturbation

    return x_adversarial.numpy()

print("FGSM function defined successfully.")

FGSM function defined successfully.


In [3]:
# Test FGSM on a small batch

epsilon = 0.01

X_test_small = X_test_lstm[:1000]
y_test_small = y_test_lstm[:1000]

X_adv_small = fgsm_attack(
    model,
    X_test_small,
    y_test_small,
    epsilon
)

print("FGSM test completed")
print("=" * 50)
print("Original shape    :", X_test_small.shape)
print("Adversarial shape :", X_adv_small.shape)
print("Epsilon           :", epsilon)

print(
    "\nMean absolute perturbation:",
    np.mean(np.abs(X_adv_small - X_test_small))
)

FGSM test completed
Original shape    : (1000, 78, 1)
Adversarial shape : (1000, 78, 1)
Epsilon           : 0.01

Mean absolute perturbation: 0.009999997433045868


In [4]:
from sklearn.metrics import accuracy_score, classification_report

# Predictions on clean samples
clean_prob_small = model.predict(
    X_test_small,
    verbose=0
)

clean_pred_small = (
    clean_prob_small >= 0.5
).astype(int).flatten()

# Predictions on adversarial samples
adv_prob_small = model.predict(
    X_adv_small,
    verbose=0
)

adv_pred_small = (
    adv_prob_small >= 0.5
).astype(int).flatten()

# Calculate accuracies
clean_accuracy_small = accuracy_score(
    y_test_small,
    clean_pred_small
)

adv_accuracy_small = accuracy_score(
    y_test_small,
    adv_pred_small
)

print("FGSM Impact on IDS")
print("=" * 50)
print(f"Clean accuracy       : {clean_accuracy_small:.4f}")
print(f"Adversarial accuracy : {adv_accuracy_small:.4f}")
print(f"Accuracy drop        : {clean_accuracy_small - adv_accuracy_small:.4f}")

FGSM Impact on IDS
Clean accuracy       : 0.9500
Adversarial accuracy : 0.8950
Accuracy drop        : 0.0550


In [5]:
# Generate FGSM adversarial examples for the full test set

epsilon = 0.01
batch_size = 1000

X_adv_batches = []

for start in range(0, len(X_test_lstm), batch_size):
    end = min(start + batch_size, len(X_test_lstm))

    X_batch = X_test_lstm[start:end]
    y_batch = y_test_lstm[start:end]

    X_adv_batch = fgsm_attack(
        model,
        X_batch,
        y_batch,
        epsilon
    )

    X_adv_batches.append(X_adv_batch)

    print(f"Processed {end}/{len(X_test_lstm)} samples")

X_test_fgsm = np.concatenate(X_adv_batches, axis=0)

print("\nFull FGSM generation complete")
print("=" * 50)
print("Adversarial test shape :", X_test_fgsm.shape)
print("Epsilon               :", epsilon)

Processed 1000/45000 samples
Processed 2000/45000 samples
Processed 3000/45000 samples
Processed 4000/45000 samples
Processed 5000/45000 samples
Processed 6000/45000 samples
Processed 7000/45000 samples
Processed 8000/45000 samples
Processed 9000/45000 samples
Processed 10000/45000 samples
Processed 11000/45000 samples
Processed 12000/45000 samples
Processed 13000/45000 samples
Processed 14000/45000 samples
Processed 15000/45000 samples
Processed 16000/45000 samples
Processed 17000/45000 samples
Processed 18000/45000 samples
Processed 19000/45000 samples
Processed 20000/45000 samples
Processed 21000/45000 samples
Processed 22000/45000 samples
Processed 23000/45000 samples
Processed 24000/45000 samples
Processed 25000/45000 samples
Processed 26000/45000 samples
Processed 27000/45000 samples
Processed 28000/45000 samples
Processed 29000/45000 samples
Processed 30000/45000 samples
Processed 31000/45000 samples
Processed 32000/45000 samples
Processed 33000/45000 samples
Processed 34000/450

In [6]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predictions on FGSM adversarial test data
adv_prob = model.predict(
    X_test_fgsm,
    batch_size=256,
    verbose=1
)

adv_pred = (
    adv_prob >= 0.5
).astype(int).flatten()

# Adversarial accuracy
adv_accuracy = accuracy_score(
    y_test_lstm,
    adv_pred
)

# Clean baseline accuracy
clean_accuracy = 0.9504

# Calculate accuracy drop
accuracy_drop = clean_accuracy - adv_accuracy

print("\nFGSM Adversarial Results")
print("=" * 50)
print(f"Clean Accuracy       : {clean_accuracy:.4f}")
print(f"FGSM Accuracy        : {adv_accuracy:.4f}")
print(f"Accuracy Drop        : {accuracy_drop:.4f}")

print("\nClassification Report")
print("=" * 50)
print(classification_report(
    y_test_lstm,
    adv_pred,
    target_names=["BENIGN", "ATTACK"],
    digits=4
))

print("\nConfusion Matrix")
print("=" * 50)
print(confusion_matrix(
    y_test_lstm,
    adv_pred
))

176/176 ━━━━━━━━━━━━━━━━━━━━ 16s 89ms/step

FGSM Adversarial Results
Clean Accuracy       : 0.9504
FGSM Accuracy        : 0.8878
Accuracy Drop        : 0.0626

Classification Report
              precision    recall  f1-score   support

      BENIGN     0.8732    0.9074    0.8900     22500
      ATTACK     0.9036    0.8682    0.8856     22500

    accuracy                         0.8878     45000
   macro avg     0.8884    0.8878    0.8878     45000
weighted avg     0.8884    0.8878    0.8878     45000


Confusion Matrix
[[20416  2084]
 [ 2965 19535]]


In [7]:
# Save FGSM adversarial test data

np.save(
    PROCESSED_DIR / "X_test_fgsm_eps001.npy",
    X_test_fgsm
)

print("FGSM adversarial test data saved successfully.")

FGSM adversarial test data saved successfully.


PGD attack

In [11]:
def pgd_attack(model, x, y, epsilon=0.01, alpha=0.002, iterations=5):
    """
    Generate adversarial examples using PGD.

    epsilon    : maximum perturbation
    alpha      : step size
    iterations : number of attack iterations
    """

    x_original = tf.convert_to_tensor(
        x, dtype=tf.float32
    )

    x_adv = tf.identity(x_original)

    y_tensor = tf.convert_to_tensor(
        y.reshape(-1, 1),
        dtype=tf.float32
    )

    for _ in range(iterations):

        with tf.GradientTape() as tape:
            tape.watch(x_adv)

            predictions = model(
                x_adv,
                training=False
            )

            loss = tf.keras.losses.binary_crossentropy(
                y_tensor,
                predictions
            )

            loss = tf.reduce_mean(loss)

        gradient = tape.gradient(loss, x_adv)

        # Gradient step
        x_adv = x_adv + alpha * tf.sign(gradient)

        # Project back into the epsilon-ball
        perturbation = tf.clip_by_value(
            x_adv - x_original,
            -epsilon,
            epsilon
        )

        x_adv = x_original + perturbation

    # Final clipping to avoid floating-point boundary overflow
    x_adv = tf.clip_by_value(
        x_adv,
        x_original - epsilon,
        x_original + epsilon
    )

    return x_adv.numpy()

print("PGD function defined successfully.")

PGD function defined successfully.


In [13]:
# Verify PGD perturbation bound using float32 values

original_float32 = X_test_small.astype(np.float32)
adversarial_float32 = X_pgd_small.astype(np.float32)

actual_perturbation = np.abs(
    adversarial_float32 - original_float32
)

print("PGD perturbation verification")
print("=" * 50)
print("Maximum absolute perturbation:",
      actual_perturbation.max())

print("Allowed epsilon:", epsilon)

print(
    "Within epsilon bound:",
    actual_perturbation.max() <= epsilon + 1e-6
)

PGD perturbation verification
Maximum absolute perturbation: 0.010000229
Allowed epsilon: 0.01
Within epsilon bound: True


In [14]:
# Evaluate PGD on the small test batch

pgd_prob_small = model.predict(
    X_pgd_small,
    verbose=0
)

pgd_pred_small = (
    pgd_prob_small >= 0.5
).astype(int).flatten()

pgd_accuracy_small = accuracy_score(
    y_test_small,
    pgd_pred_small
)

clean_accuracy_small = accuracy_score(
    y_test_small,
    clean_pred_small
)

print("PGD Impact on IDS")
print("=" * 50)
print(f"Clean accuracy : {clean_accuracy_small:.4f}")
print(f"PGD accuracy   : {pgd_accuracy_small:.4f}")
print(
    f"Accuracy drop  : "
    f"{clean_accuracy_small - pgd_accuracy_small:.4f}"
)

PGD Impact on IDS
Clean accuracy : 0.9500
PGD accuracy   : 0.8860
Accuracy drop  : 0.0640


In [15]:
# Generate PGD adversarial examples for the full test set

epsilon = 0.01
alpha = 0.002
iterations = 5
batch_size = 1000

X_pgd_batches = []

for start in range(0, len(X_test_lstm), batch_size):
    end = min(start + batch_size, len(X_test_lstm))

    X_batch = X_test_lstm[start:end]
    y_batch = y_test_lstm[start:end]

    X_pgd_batch = pgd_attack(
        model,
        X_batch,
        y_batch,
        epsilon=epsilon,
        alpha=alpha,
        iterations=iterations
    )

    X_pgd_batches.append(X_pgd_batch)

    print(f"Processed {end}/{len(X_test_lstm)} samples")

X_test_pgd = np.concatenate(
    X_pgd_batches,
    axis=0
)

print("\nFull PGD generation complete")
print("=" * 50)
print("Adversarial test shape :", X_test_pgd.shape)
print("Epsilon               :", epsilon)
print("Step size             :", alpha)
print("Iterations            :", iterations)

Processed 1000/45000 samples
Processed 2000/45000 samples
Processed 3000/45000 samples
Processed 4000/45000 samples
Processed 5000/45000 samples
Processed 6000/45000 samples
Processed 7000/45000 samples
Processed 8000/45000 samples
Processed 9000/45000 samples
Processed 10000/45000 samples
Processed 11000/45000 samples
Processed 12000/45000 samples
Processed 13000/45000 samples
Processed 14000/45000 samples
Processed 15000/45000 samples
Processed 16000/45000 samples
Processed 17000/45000 samples
Processed 18000/45000 samples
Processed 19000/45000 samples
Processed 20000/45000 samples
Processed 21000/45000 samples
Processed 22000/45000 samples
Processed 23000/45000 samples
Processed 24000/45000 samples
Processed 25000/45000 samples
Processed 26000/45000 samples
Processed 27000/45000 samples
Processed 28000/45000 samples
Processed 29000/45000 samples
Processed 30000/45000 samples
Processed 31000/45000 samples
Processed 32000/45000 samples
Processed 33000/45000 samples
Processed 34000/450

In [16]:
# Evaluate the LSTM on full PGD adversarial test data

pgd_prob = model.predict(
    X_test_pgd,
    batch_size=256,
    verbose=1
)

pgd_pred = (
    pgd_prob >= 0.5
).astype(int).flatten()

# PGD accuracy
pgd_accuracy = accuracy_score(
    y_test_lstm,
    pgd_pred
)

# Clean baseline
clean_accuracy = 0.9504

# Accuracy drop
pgd_accuracy_drop = clean_accuracy - pgd_accuracy

print("\nPGD Adversarial Results")
print("=" * 50)
print(f"Clean Accuracy       : {clean_accuracy:.4f}")
print(f"PGD Accuracy         : {pgd_accuracy:.4f}")
print(f"Accuracy Drop        : {pgd_accuracy_drop:.4f}")

print("\nClassification Report")
print("=" * 50)

print(classification_report(
    y_test_lstm,
    pgd_pred,
    target_names=["BENIGN", "ATTACK"],
    digits=4
))

print("\nConfusion Matrix")
print("=" * 50)

print(confusion_matrix(
    y_test_lstm,
    pgd_pred
))

176/176 ━━━━━━━━━━━━━━━━━━━━ 14s 81ms/step

PGD Adversarial Results
Clean Accuracy       : 0.9504
PGD Accuracy         : 0.8783
Accuracy Drop        : 0.0721

Classification Report
              precision    recall  f1-score   support

      BENIGN     0.8589    0.9053    0.8815     22500
      ATTACK     0.8999    0.8513    0.8749     22500

    accuracy                         0.8783     45000
   macro avg     0.8794    0.8783    0.8782     45000
weighted avg     0.8794    0.8783    0.8782     45000


Confusion Matrix
[[20370  2130]
 [ 3346 19154]]


In [17]:
# Save PGD adversarial test data

np.save(
    PROCESSED_DIR / "X_test_pgd_eps001.npy",
    X_test_pgd
)

print("PGD adversarial test data saved successfully.")

PGD adversarial test data saved successfully.


Deepfool attack

In [30]:
# Prepare a small test set for DeepFool

X_deepfool_test = X_test_lstm[:100]
y_deepfool_test = y_test_lstm[:100]

print("DeepFool test data prepared")
print("=" * 50)
print("Test data shape :", X_deepfool_test.shape)
print("Test labels     :", y_deepfool_test.shape)

DeepFool test data prepared
Test data shape : (100, 78, 1)
Test labels     : (100,)


In [34]:
def deepfool_attack(model, x, max_iterations=20, overshoot=0.02):
    """
    DeepFool attack for a binary sigmoid classifier.

    Returns:
        x_adv           : adversarial samples
        success         : whether prediction changed
        iterations_used : iterations required
    """

    x_original = tf.convert_to_tensor(x, dtype=tf.float32)
    x_adv = tf.identity(x_original)

    n_samples = x_original.shape[0]

    # Original predictions
    original_probability = model(
        x_original,
        training=False
    )

    # Force shape to (n_samples,)
    original_probability = tf.reshape(
        original_probability,
        [n_samples]
    )

    original_class = tf.cast(
        original_probability >= 0.5,
        tf.int32
    )

    active = tf.ones(
        [n_samples],
        dtype=tf.bool
    )

    iterations_used = np.zeros(
        n_samples,
        dtype=np.int32
    )

    for iteration in range(max_iterations):

        # Gradient of the logit
        with tf.GradientTape() as tape:
            tape.watch(x_adv)

            probability = model(
                x_adv,
                training=False
            )

            probability = tf.reshape(
                probability,
                [n_samples]
            )

            probability = tf.clip_by_value(
                probability,
                1e-6,
                1 - 1e-6
            )

            logit = tf.math.log(
                probability / (1.0 - probability)
            )

        gradient = tape.gradient(
            logit,
            x_adv
        )

        # Flatten gradient for each sample
        gradient_flat = tf.reshape(
            gradient,
            [n_samples, -1]
        )

        # Gradient norm squared for each sample
        gradient_norm_squared = tf.reduce_sum(
            tf.square(gradient_flat),
            axis=1
        )

        gradient_norm_squared = tf.maximum(
            gradient_norm_squared,
            1e-12
        )

        # DeepFool step for each sample
        step = (
            -logit
            / gradient_norm_squared
        )

        step = tf.reshape(
            step,
            [n_samples, 1, 1]
        )

        perturbation = (
            step * gradient
        )

        # Only update samples that are not fooled yet
        active_mask = tf.cast(
            active[:, None, None],
            tf.float32
        )

        x_adv = x_adv + (
            (1.0 + overshoot)
            * perturbation
            * active_mask
        )

        # Check new predictions
        new_probability = model(
            x_adv,
            training=False
        )

        new_probability = tf.reshape(
            new_probability,
            [n_samples]
        )

        new_class = tf.cast(
            new_probability >= 0.5,
            tf.int32
        )

        # Samples whose class changed
        fooled = tf.not_equal(
            new_class,
            original_class
        )

        newly_fooled = tf.logical_and(
            active,
            fooled
        )

        # Record iteration
        newly_fooled_np = newly_fooled.numpy()

        iterations_used[newly_fooled_np] = (
            iteration + 1
        )

        # Remove fooled samples
        active = tf.logical_and(
            active,
            tf.logical_not(fooled)
        )

        # Stop if everything is fooled
        if not bool(tf.reduce_any(active).numpy()):
            break

    # Final predictions
    final_probability = model(
        x_adv,
        training=False
    )

    final_probability = tf.reshape(
        final_probability,
        [n_samples]
    )

    final_class = tf.cast(
        final_probability >= 0.5,
        tf.int32
    )

    success = tf.not_equal(
        final_class,
        original_class
    )

    return (
        x_adv.numpy(),
        success.numpy(),
        iterations_used
    )

In [35]:
# Run DeepFool on the 100-sample test set

X_deepfool_adv, deepfool_success, deepfool_iterations = deepfool_attack(
    model,
    X_deepfool_test,
    max_iterations=20,
    overshoot=0.02
)

print("DeepFool attack completed")
print("=" * 50)
print("Adversarial data shape :", X_deepfool_adv.shape)
print("Successful attacks     :", np.sum(deepfool_success))
print("Failed attacks         :", np.sum(~deepfool_success))
print("Success rate            :", np.mean(deepfool_success))

DeepFool attack completed
Adversarial data shape : (100, 78, 1)
Successful attacks     : 100
Failed attacks         : 0
Success rate            : 1.0


In [36]:
# Compare clean and DeepFool predictions

clean_prob = model.predict(
    X_deepfool_test,
    verbose=0
).reshape(-1)

deepfool_prob = model.predict(
    X_deepfool_adv,
    verbose=0
).reshape(-1)

clean_pred = (clean_prob >= 0.5).astype(int)
deepfool_pred = (deepfool_prob >= 0.5).astype(int)

clean_accuracy = np.mean(
    clean_pred == y_deepfool_test
)

deepfool_accuracy = np.mean(
    deepfool_pred == y_deepfool_test
)

prediction_changes = np.sum(
    clean_pred != deepfool_pred
)

print("DeepFool Prediction Analysis")
print("=" * 50)
print(f"Clean accuracy          : {clean_accuracy:.4f}")
print(f"DeepFool accuracy       : {deepfool_accuracy:.4f}")
print(f"Accuracy drop           : {clean_accuracy - deepfool_accuracy:.4f}")
print(f"Prediction changes     : {prediction_changes}")
print(f"Attack success rate     : {np.mean(deepfool_success):.4f}")

DeepFool Prediction Analysis
Clean accuracy          : 0.9300
DeepFool accuracy       : 0.0700
Accuracy drop           : 0.8600
Prediction changes     : 100
Attack success rate     : 1.0000


In [37]:
# Analyze DeepFool perturbation magnitude

perturbation = X_deepfool_adv - X_deepfool_test

# Absolute perturbation
abs_perturbation = np.abs(perturbation)

# L2 norm for each sample
perturbation_flat = perturbation.reshape(
    perturbation.shape[0],
    -1
)

l2_norms = np.linalg.norm(
    perturbation_flat,
    axis=1
)

# L-infinity norm for each sample
linf_norms = np.max(
    np.abs(perturbation_flat),
    axis=1
)

print("DeepFool Perturbation Analysis")
print("=" * 50)

print(f"Mean L2 perturbation : {np.mean(l2_norms):.6f}")
print(f"Min L2 perturbation  : {np.min(l2_norms):.6f}")
print(f"Max L2 perturbation  : {np.max(l2_norms):.6f}")

print()

print(f"Mean L-inf perturbation : {np.mean(linf_norms):.6f}")
print(f"Min L-inf perturbation  : {np.min(linf_norms):.6f}")
print(f"Max L-inf perturbation  : {np.max(linf_norms):.6f}")

print()

print("L2 percentiles:")
print("25% :", np.percentile(l2_norms, 25))
print("50% :", np.percentile(l2_norms, 50))
print("75% :", np.percentile(l2_norms, 75))
print("90% :", np.percentile(l2_norms, 90))

DeepFool Perturbation Analysis
Mean L2 perturbation : 2.094648
Min L2 perturbation  : 0.014416
Max L2 perturbation  : 20.324622

Mean L-inf perturbation : 0.848572
Min L-inf perturbation  : 0.005276
Max L-inf perturbation  : 9.828969

L2 percentiles:
25% : 0.32625248834947823
50% : 0.5999229830481019
75% : 2.3273353529959366
90% : 5.05136342548353


In [38]:
# Check prediction probabilities before and after DeepFool

print("Sample-wise DeepFool Check")
print("=" * 70)

for i in range(10):
    print(
        f"Sample {i:2d} | "
        f"Clean P(Attack): {clean_prob[i]:.4f} | "
        f"DeepFool P(Attack): {deepfool_prob[i]:.4f} | "
        f"Clean class: {clean_pred[i]} | "
        f"New class: {deepfool_pred[i]} | "
        f"L2: {l2_norms[i]:.4f} | "
        f"Iterations: {deepfool_iterations[i]}"
    )

Sample-wise DeepFool Check
Sample  0 | Clean P(Attack): 0.9995 | DeepFool P(Attack): 0.1781 | Clean class: 1 | New class: 0 | L2: 1.5732 | Iterations: 1
Sample  1 | Clean P(Attack): 0.9878 | DeepFool P(Attack): 0.0446 | Clean class: 1 | New class: 0 | L2: 0.4246 | Iterations: 1
Sample  2 | Clean P(Attack): 0.0004 | DeepFool P(Attack): 0.9700 | Clean class: 0 | New class: 1 | L2: 1.0552 | Iterations: 2
Sample  3 | Clean P(Attack): 0.0089 | DeepFool P(Attack): 0.5003 | Clean class: 0 | New class: 1 | L2: 0.3422 | Iterations: 3
Sample  4 | Clean P(Attack): 0.1625 | DeepFool P(Attack): 0.5106 | Clean class: 0 | New class: 1 | L2: 0.1301 | Iterations: 2
Sample  5 | Clean P(Attack): 0.9887 | DeepFool P(Attack): 0.4829 | Clean class: 1 | New class: 0 | L2: 0.2286 | Iterations: 1
Sample  6 | Clean P(Attack): 0.0336 | DeepFool P(Attack): 0.5025 | Clean class: 0 | New class: 1 | L2: 0.4515 | Iterations: 4
Sample  7 | Clean P(Attack): 0.7003 | DeepFool P(Attack): 0.4924 | Clean class: 1 | New cla

In [40]:
# Run DeepFool on the full test set in batches

BATCH_SIZE = 500
N_SAMPLES = len(X_test_lstm)

deepfool_batches = []
success_batches = []
iteration_batches = []

print("Starting DeepFool attack")
print("=" * 60)
print(f"Total samples : {N_SAMPLES}")
print(f"Batch size    : {BATCH_SIZE}")
print()

for start in range(0, N_SAMPLES, BATCH_SIZE):

    end = min(start + BATCH_SIZE, N_SAMPLES)

    X_batch = X_test_lstm[start:end]

    X_adv_batch, success_batch, iterations_batch = deepfool_attack(
        model,
        X_batch,
        max_iterations=20,
        overshoot=0.02
    )

    deepfool_batches.append(X_adv_batch)
    success_batches.append(success_batch)
    iteration_batches.append(iterations_batch)

    completed = end
    successful_so_far = np.sum(
        np.concatenate(success_batches)
    )

    print(
        f"Progress: {completed:,} / {N_SAMPLES:,} "
        f"({completed / N_SAMPLES * 100:.1f}%) | "
        f"Successful: {successful_so_far:,}"
    )

# Combine all batches
X_deepfool_full = np.concatenate(
    deepfool_batches,
    axis=0
)

deepfool_success_full = np.concatenate(
    success_batches,
    axis=0
)

deepfool_iterations_full = np.concatenate(
    iteration_batches,
    axis=0
)

print()
print("=" * 60)
print("DeepFool attack completed")
print("=" * 60)
print("Adversarial data shape :", X_deepfool_full.shape)
print("Successful attacks     :", np.sum(deepfool_success_full))
print("Failed attacks         :", np.sum(~deepfool_success_full))
print("Success rate           :", np.mean(deepfool_success_full))

Starting DeepFool attack
Total samples : 45000
Batch size    : 500

Progress: 500 / 45,000 (1.1%) | Successful: 498
Progress: 1,000 / 45,000 (2.2%) | Successful: 996
Progress: 1,500 / 45,000 (3.3%) | Successful: 1,495
Progress: 2,000 / 45,000 (4.4%) | Successful: 1,992
Progress: 2,500 / 45,000 (5.6%) | Successful: 2,488
Progress: 3,000 / 45,000 (6.7%) | Successful: 2,987
Progress: 3,500 / 45,000 (7.8%) | Successful: 3,486
Progress: 4,000 / 45,000 (8.9%) | Successful: 3,985
Progress: 4,500 / 45,000 (10.0%) | Successful: 4,484
Progress: 5,000 / 45,000 (11.1%) | Successful: 4,980
Progress: 5,500 / 45,000 (12.2%) | Successful: 5,475
Progress: 6,000 / 45,000 (13.3%) | Successful: 5,972
Progress: 6,500 / 45,000 (14.4%) | Successful: 6,470
Progress: 7,000 / 45,000 (15.6%) | Successful: 6,968
Progress: 7,500 / 45,000 (16.7%) | Successful: 7,466
Progress: 8,000 / 45,000 (17.8%) | Successful: 7,963
Progress: 8,500 / 45,000 (18.9%) | Successful: 8,461
Progress: 9,000 / 45,000 (20.0%) | Successful

In [41]:
# Full DeepFool prediction analysis

clean_prob_full = model.predict(
    X_test_lstm,
    verbose=0
).reshape(-1)

deepfool_prob_full = model.predict(
    X_deepfool_full,
    verbose=0
).reshape(-1)

clean_pred_full = (
    clean_prob_full >= 0.5
).astype(int)

deepfool_pred_full = (
    deepfool_prob_full >= 0.5
).astype(int)

clean_accuracy_full = np.mean(
    clean_pred_full == y_test_lstm
)

deepfool_accuracy_full = np.mean(
    deepfool_pred_full == y_test_lstm
)

prediction_changes_full = np.sum(
    clean_pred_full != deepfool_pred_full
)

print("Full DeepFool Prediction Analysis")
print("=" * 55)
print(f"Clean accuracy       : {clean_accuracy_full:.4f}")
print(f"DeepFool accuracy    : {deepfool_accuracy_full:.4f}")
print(f"Accuracy drop        : {clean_accuracy_full - deepfool_accuracy_full:.4f}")
print(f"Prediction changes   : {prediction_changes_full}")
print(f"Attack success rate  : {np.mean(deepfool_success_full):.4f}")

Full DeepFool Prediction Analysis
Clean accuracy       : 0.9504
DeepFool accuracy    : 0.0537
Accuracy drop        : 0.8967
Prediction changes   : 44809
Attack success rate  : 0.9957


In [42]:
# Full DeepFool perturbation analysis

perturbation_full = (
    X_deepfool_full - X_test_lstm
)

perturbation_flat_full = perturbation_full.reshape(
    perturbation_full.shape[0],
    -1
)

l2_norms_full = np.linalg.norm(
    perturbation_flat_full,
    axis=1
)

linf_norms_full = np.max(
    np.abs(perturbation_flat_full),
    axis=1
)

print("Full DeepFool Perturbation Analysis")
print("=" * 55)

print(f"Mean L2 perturbation : {np.mean(l2_norms_full):.6f}")
print(f"Median L2 perturbation : {np.median(l2_norms_full):.6f}")
print(f"Min L2 perturbation  : {np.min(l2_norms_full):.6f}")
print(f"Max L2 perturbation  : {np.max(l2_norms_full):.6f}")

print()

print(f"Mean L-inf perturbation : {np.mean(linf_norms_full):.6f}")
print(f"Median L-inf perturbation : {np.median(linf_norms_full):.6f}")
print(f"Min L-inf perturbation  : {np.min(linf_norms_full):.6f}")
print(f"Max L-inf perturbation  : {np.max(linf_norms_full):.6f}")

print()

print("L2 percentiles:")
print(f"25% : {np.percentile(l2_norms_full, 25):.6f}")
print(f"50% : {np.percentile(l2_norms_full, 50):.6f}")
print(f"75% : {np.percentile(l2_norms_full, 75):.6f}")
print(f"90% : {np.percentile(l2_norms_full, 90):.6f}")

Full DeepFool Perturbation Analysis
Mean L2 perturbation : 2.195557
Median L2 perturbation : 1.076561
Min L2 perturbation  : 0.000000
Max L2 perturbation  : 37.874400

Mean L-inf perturbation : 0.787741
Median L-inf perturbation : 0.404726
Min L-inf perturbation  : 0.000000
Max L-inf perturbation  : 20.937454

L2 percentiles:
25% : 0.333372
50% : 1.076561
75% : 2.949787
90% : 5.729356


In [43]:
# Save DeepFool adversarial examples and attack information

np.save(
    PROCESSED_DIR / "X_test_deepfool.npy",
    X_deepfool_full
)

np.save(
    PROCESSED_DIR / "deepfool_success.npy",
    deepfool_success_full
)

np.save(
    PROCESSED_DIR / "deepfool_iterations.npy",
    deepfool_iterations_full
)

print("DeepFool results saved successfully.")
print("=" * 50)
print("X_test_deepfool.npy")
print("deepfool_success.npy")
print("deepfool_iterations.npy")

DeepFool results saved successfully.
X_test_deepfool.npy
deepfool_success.npy
deepfool_iterations.npy


In [44]:
# Check DeepFool success on originally correctly classified samples

correct_clean = (
    clean_pred_full == y_test_lstm
)

correct_clean_count = np.sum(correct_clean)

successful_on_correct = np.sum(
    deepfool_success_full & correct_clean
)

print("DeepFool Correctly-Classified Sample Analysis")
print("=" * 60)

print(f"Originally correct samples : {correct_clean_count:,}")
print(f"Originally incorrect       : {len(y_test_lstm) - correct_clean_count:,}")

print(
    f"Successful attacks on originally correct samples : "
    f"{successful_on_correct:,}"
)

print(
    f"Success rate on originally correct samples        : "
    f"{successful_on_correct / correct_clean_count:.4f}"
)

DeepFool Correctly-Classified Sample Analysis
Originally correct samples : 42,768
Originally incorrect       : 2,232
Successful attacks on originally correct samples : 42,577
Success rate on originally correct samples        : 0.9955


In [45]:
# Analyze DeepFool iterations

successful_iterations = deepfool_iterations_full[
    deepfool_success_full
]

print("DeepFool Iteration Analysis")
print("=" * 60)

print(f"Mean iterations   : {np.mean(successful_iterations):.2f}")
print(f"Median iterations : {np.median(successful_iterations):.2f}")
print(f"Maximum iterations: {np.max(successful_iterations)}")

print()
print("Iteration distribution:")
for i in range(1, 21):
    count = np.sum(successful_iterations == i)
    if count > 0:
        print(f"{i:2d} iterations : {count:,}")

DeepFool Iteration Analysis
Mean iterations   : 2.73
Median iterations : 2.00
Maximum iterations: 20

Iteration distribution:
 1 iterations : 14,711
 2 iterations : 11,717
 3 iterations : 7,428
 4 iterations : 4,622
 5 iterations : 2,879
 6 iterations : 1,188
 7 iterations : 746
 8 iterations : 399
 9 iterations : 255
10 iterations : 215
11 iterations : 133
12 iterations : 88
13 iterations : 85
14 iterations : 86
15 iterations : 67
16 iterations : 54
17 iterations : 61
18 iterations : 38
19 iterations : 20
20 iterations : 13


In [46]:
np.save(
    PROCESSED_DIR / "X_test_deepfool.npy",
    X_deepfool_full
)

np.save(
    PROCESSED_DIR / "deepfool_success.npy",
    deepfool_success_full
)

np.save(
    PROCESSED_DIR / "deepfool_iterations.npy",
    deepfool_iterations_full
)

print("DeepFool results saved successfully.")

DeepFool results saved successfully.
